# N30 — RAG Agent

The RAG Agent answers one question: **what does the FIA regulation say about this?**

It builds a retrieval-augmented generation pipeline over the FIA Sporting Regulations (2023–2025). Given a natural-language query from the Strategy Orchestrator, it retrieves the most relevant regulation chunks from a local Qdrant vector store and returns them as structured `RegulationContext` objects — with article references and relevance scores.

This agent is event-driven: N31 activates it when N27 reports `sc_prob > 0.3` (safety car rules become relevant) or when N29 detects a `PROBLEM` or `ORDER` radio intent (driver or team directive that may have a regulatory dimension).

## Pipeline position

<pre>
query (natural language, from N31 Orchestrator)
    │
    ▼
N30 — RAG Agent  (LangGraph ReAct)
    │
    ├── query_rag_tool         dense retrieval from Qdrant
    │
    └── RegulationContext
            ├── chunks[]       relevant regulation text passages
            ├── articles[]     article/section references (e.g. Art. 48.3)
            └── answer         LLM-synthesised plain-language summary
</pre>

## Stack
- **Embedding model** — `BAAI/bge-m3` (sentence-transformers, 1024-dim, RTX 5070 GPU)
- **Vector store** — Qdrant local (on-disk, no Docker required)
- **Documents** — FIA Sporting Regulations 2023–2025 (PDF → text → chunks)
- **Chunking** — 512-character overlapping windows (overlap 64 characters)
- **Retrieval** — cosine similarity top-k, k=5 per query

## Steps
- **Step 0** — Setup, imports & health check
- **Step 1** — Index overview (what's been indexed and how)
- **Step 2** — Sample chunk payloads (metadata sanity check)
- **Step 3** — `query_rag_tool` direct invocation demo
- **Step 4** — `RegulationContext` dataclass + `run_rag_agent` entry point
- **Step 5** — LangGraph ReAct agent (`create_react_agent`)
- **Step 6** — Demo queries across all regulatory domains
- **Step 7** — Export agent config

---

In [ ]:
# ── Step 0: Setup & imports ───────────────────────────────────────────────────
# pip install qdrant-client sentence-transformers langchain langchain-anthropic langgraph pypdf

import sys
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any

# ── Repo root resolution (works from any cwd) ─────────────────────────────────
repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── RAG module (src/rag/retriever.py) ─────────────────────────────────────────
from src.rag.retriever import (
    RagRetriever,
    RegulationChunk,
    get_retriever,
    query_rag_tool,
    CFG,
)

# ── LangGraph + LLM ───────────────────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

OUTPUTS_DIR = repo_root / "notebooks" / "agents" / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Health check — confirms index is ready ────────────────────────────────────
retriever = get_retriever()
hc        = retriever.health_check()

print(f"Collection   : {hc['collection']}")
print(f"Vector count : {hc['vector_count']:,}")
print(f"Model        : {hc['embedding_model']}")
print(f"Storage      : {hc['qdrant_path']}")

---

## Step 1 — Index overview

The vector index was built offline by running:

```bash
python scripts/download_fia_pdfs.py    # download FIA PDFs → data/rag/documents/
python scripts/build_rag_index.py      # chunk + embed + upsert → data/rag/qdrant_local/
```

`build_rag_index.py` ingests every `*.pdf` in `data/rag/documents/`, splits each document into overlapping 512-character windows (64-character overlap), embeds with **BGE-M3** (1024-dim), and upserts into a local Qdrant collection. The pipeline is idempotent — chunks already indexed by SHA-256 hash are skipped on re-runs.

This step just inspects what is already in the index.

In [ ]:
# ── Step 1: Inspect what's been indexed ──────────────────────────────────────
docs_dir = CFG.qdrant_path.parent / "documents"   # data/rag/documents/
pdfs     = sorted(docs_dir.glob("*.pdf"))

print("PDFs in data/rag/documents/")
print("─" * 50)
for pdf in pdfs:
    size_kb = pdf.stat().st_size // 1024
    print(f"  {pdf.name:<45}  {size_kb:>5} KB")

print()
print(f"Total PDFs             : {len(pdfs)}")
print(f"Chunks indexed (Qdrant): {hc['vector_count']:,}")

---

## Step 2 — Sample chunk payloads

A quick sanity check: scroll a few raw payloads directly from Qdrant to confirm the metadata fields (`article`, `doc_type`, `year`, `section_title`) were stored correctly during indexing and match what `RegulationChunk` expects at query time.

In [ ]:
# ── Step 2: Preview stored chunk payloads ────────────────────────────────────
from qdrant_client import QdrantClient as _QdrantClient

_qdrant = _QdrantClient(path=str(CFG.qdrant_path))

sample_points, _ = _qdrant.scroll(
    collection_name=CFG.collection_name,
    limit=3,
    with_payload=True,
    with_vectors=False,
)

for i, pt in enumerate(sample_points, start=1):
    p = pt.payload
    print(f"── Chunk {i} {'─' * 52}")
    print(f"  doc_type      : {p.get('doc_type')}")
    print(f"  year          : {p.get('year')}")
    print(f"  article       : {p.get('article')}")
    print(f"  section_title : {p.get('section_title')}")
    print(f"  text preview  : {p.get('text', '')[:120].replace(chr(10), ' ')!r}")
    print()

---

## Step 3 — `query_rag_tool` direct demo

`query_rag_tool` is the `@tool`-decorated wrapper in `src/rag/retriever.py`. It encodes the query with BGE-M3, searches Qdrant by cosine similarity, and formats the top-k chunks as a plain string for the LLM.

Run a few queries directly here to sanity-check retrieval quality before wiring the tool into the agent in Step 5.

In [ ]:
# ── Step 3: Direct query_rag_tool invocation ──────────────────────────────────
# query_rag_tool is defined in src/rag/retriever.py as a @tool-decorated function.
# Run a few queries directly to verify retrieval quality before wiring into the agent.

queries = [
    "What are the rules when a safety car is deployed?",
    "Pit lane speed limit during a race",
    "Penalties for causing a collision",
    "Mandatory tyre compounds during the race",
]

for q in queries:
    print(f"QUERY: {q}")
    print("─" * 60)
    result = query_rag_tool.invoke({"question": q})
    print(result[:450])
    print("..." if len(result) > 450 else "")
    print()

---

## Step 4 — `RegulationContext` + `run_rag_agent`

The orchestrator (N31) expects a structured response, not a raw string. `RegulationContext` packages the retrieved passages alongside a plain-language summary produced by the LLM — so N31 can both cite sources and act on a concise answer.

`run_rag_agent(question, agent)` is the public entry point: it calls the LangGraph agent, collects the final message, and re-queries the retriever to populate the structured chunk fields.

In [ ]:
# ── Step 4: RegulationContext + run_rag_agent entry point ─────────────────────

@dataclass
class RegulationContext:
    """Structured output returned by the RAG agent for a single query.

    Bundles the LLM's plain-language summary with the source regulation chunks
    it was derived from, so downstream agents can both act on a concise answer
    and cite specific FIA articles without re-reading the raw passages.

    Attributes:
        question: The original natural-language question that triggered this
                  lookup. Stored so the orchestrator can log which queries
                  were issued and detect duplicate lookups within a race lap.
        answer:   LLM-generated summary of the relevant regulation articles —
                  one to three sentences, enough for the Strategy Orchestrator
                  to decide whether a proposed action is legal without reading
                  the full passage.
        chunks:   The raw ``RegulationChunk`` objects returned by
                  ``query_rag_tool``. Kept alongside the summary so callers
                  can filter by article range, year, or doc_type when the
                  answer is ambiguous.
        articles: Deduplicated list of article references from the chunks
                  (e.g. ``["Article 48.3", "Article 55.1"]``). Used by N31
                  to attach precise citations to strategy log entries.
    """

    question: str
    answer:   str
    chunks:   list[RegulationChunk] = field(default_factory=list)
    articles: list[str]             = field(default_factory=list)

    def __repr__(self) -> str:
        return (
            f"RegulationContext("
            f"articles={self.articles}, "
            f"answer={self.answer[:80]!r}...)"
        )


def run_rag_agent(question: str, agent) -> RegulationContext:
    """Run the RAG LangGraph agent for a single regulation question.

    Invokes the ReAct agent, extracts the final answer message, and re-queries
    the retriever to populate the ``RegulationContext`` with structured chunks
    (the agent only returns a string, not chunk objects).

    Args:
        question: Natural-language regulation question from the orchestrator.
                  Examples: "Is overtaking under VSC allowed?",
                  "What is the minimum pit stop time?".
        agent:    A compiled LangGraph ReAct graph. Passed in rather than
                  instantiated here so the caller controls model selection
                  and the same agent instance can be reused across calls
                  without reloading the LLM on each invocation.

    Returns:
        ``RegulationContext`` with the LLM answer, source chunks, and
        deduplicated article references extracted from the chunk metadata.
    """
    result   = agent.invoke({"messages": [HumanMessage(content=question)]})
    answer   = result["messages"][-1].content
    chunks   = retriever.query(question)
    articles = list(dict.fromkeys(c.article for c in chunks if c.article))

    return RegulationContext(
        question=question,
        answer=answer,
        chunks=chunks,
        articles=articles,
    )


# Quick smoke test — no LLM call yet
ctx_test = RegulationContext(
    question="test",
    answer="test answer",
    chunks=[],
    articles=["Article 48.3"],
)
print(repr(ctx_test))

---

## Step 5 — LangGraph ReAct agent

The agent follows the standard LangGraph ReAct loop:

1. **Reason** — the LLM reads the question and decides what to ask the tool
2. **Act** — calls `query_rag_tool` with a focused sub-question
3. **Observe** — reads the regulation passages returned by Qdrant
4. **Answer** — synthesises a concise answer with article citations

The system prompt instructs the model to always cite article numbers and to flag when a question spans multiple regulatory domains.

In [ ]:
# ── Step 5: Build the LangGraph ReAct agent ───────────────────────────────────
SYSTEM_PROMPT = """You are an FIA Formula 1 regulation expert agent.
You have access to a tool that retrieves passages from the official FIA Sporting
Regulations (2023–2025). When asked a regulation question:
1. Call query_rag_tool with a precise, focused question.
2. Read the retrieved passages carefully.
3. Answer in 2–3 sentences, citing the exact article numbers (e.g. "Article 48.3").
4. If the question spans multiple articles, cite each one.
5. If no relevant passage is found, say "The regulation does not cover this case."

Always prefer the most recent regulation year (2025) unless the question specifies otherwise.
"""

MODEL_NAME = "claude-opus-4-6"   # swap to claude-sonnet-4-6 for faster/cheaper runs

llm = ChatAnthropic(model=MODEL_NAME, temperature=0)

rag_react_agent = create_react_agent(
    model=llm,
    tools=[query_rag_tool],
    prompt=SYSTEM_PROMPT,
)

print(f"RAG agent created — model : {MODEL_NAME}")
print(f"Tools                     : {[query_rag_tool.name]}")

---

## Step 6 — Demo queries

Test the full pipeline — LLM + retrieval — across the four regulatory domains the Strategy Orchestrator relies on most:

- **Safety car / VSC** deployment procedures
- **Pit lane** speed limits and rules
- **Tyre** allocation and mandatory compound rules
- **Sanctions** for common on-track incidents

In [ ]:
# ── Step 6: Demo queries across all regulatory domains ────────────────────────
demo_questions = [
    "What must a driver do when the safety car is deployed?",
    "What is the speed limit in the pit lane during a race?",
    "What tyre compounds must a driver use during a dry race?",
    "What penalty applies for causing a collision with another car?",
]

demo_results: list[RegulationContext] = []

for q in demo_questions:
    print(f"\n{'═' * 65}")
    print(f"Q: {q}")
    print('═' * 65)
    ctx = run_rag_agent(q, rag_react_agent)
    print(f"\nA: {ctx.answer}")
    print(f"\nArticles cited : {ctx.articles}")
    print(f"Chunks used    : {len(ctx.chunks)}")
    demo_results.append(ctx)

---

## Step 7 — Export agent config

Save the agent configuration to `data/models/agents/rag_agent_config_v1.json`. N31 reads this file at startup to instantiate the RAG agent with the same parameters used here.

In [ ]:
# ── Step 7: Export agent config ───────────────────────────────────────────────
EXPORT_DIR = repo_root / "data" / "models" / "agents"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

rag_config = {
    "agent_name":        "rag_agent",
    "version":           "v1",
    "model":             MODEL_NAME,
    "embedding_model":   CFG.embedding_model,
    "collection_name":   CFG.collection_name,
    "top_k":             CFG.top_k,
    "qdrant_path":       str(CFG.qdrant_path),
    "documents_indexed": hc["vector_count"],
    "regulation_years":  [2023, 2024, 2025],
    "regulation_domains": [
        "safety_car_vsc",
        "pit_lane_procedures",
        "tyre_allocation",
        "sanctions_penalties",
        "blue_flags_overtaking",
        "team_orders",
    ],
    "entry_point": "run_rag_agent(question, rag_react_agent) -> RegulationContext",
    "tool":        "query_rag_tool",
    "demo_queries": [
        {"question": ctx.question, "articles": ctx.articles}
        for ctx in demo_results
    ],
}

config_path = EXPORT_DIR / "rag_agent_config_v1.json"
config_path.write_text(
    json.dumps(rag_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Config saved → {config_path.relative_to(repo_root)}")
print(json.dumps(rag_config, indent=2))